# Scientific Validity of ASI for Binary Mean Estimation

This notebook provides empirical evidence that GLIDE's **Active Statistical Inference (ASI)** implementation is statistically valid.

**Setup:** We estimate the mean of a binary outcome (e.g., the pass rate of an LLM-as-a-judge evaluation). We have:
- A pool of `N_TOTAL` records, each with a proxy label (`y_proxy`) and an oracle proxy error (`rms_error`) that quantifies how unreliable the proxy is for each individual record
- A labeling budget of `N_LABELED` items: we can reveal the true label (`y_true`) for only a fraction of the pool

**The key question is how to spend the labeling budget.** Two strategies compete:
- **Uniform sampling (PPI++):** label records uniformly at random — all items equally likely to be selected
- **Oracle sampling (ASI):** label records preferentially where the proxy is least reliable — items with high `rms_error` are selected with higher probability $\pi_i \propto \text{rms\_error}_i$

ASI corrects for this non-uniform selection via **Inverse Probability Weighting (IPW)**, yielding confidence intervals that are:
1. **Valid** : they cover the true mean at the specified rate regardless of the sampling rule
2. **More efficient** : oracle sampling concentrates the labeling budget on uncertain items, producing shorter intervals than uniform PPI++

We test these two claims empirically across a range of proxy/true correlation levels.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from glide.core.dataset import Dataset
from glide.core.simulated_datasets import generate_binary_dataset_with_oracle_sampling
from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator, PPIMeanEstimator

## Experiment Parameters

We fix all parameters up front so every section of this notebook uses a consistent setup. We define :

- `CONFIDENCE_LEVEL` : the confidence level at which we will compute confidence intervals.

- `N_TOTAL` : the total number of records in the pool. Every record carries a proxy prediction and an oracle proxy error (`rms_error`).

- `N_LABELED` : the labeling budget. For PPI++ this is the exact number of uniformly selected records; for ASI this is the **expected** number of oracle-selected records (Bernoulli sampling).

- `TRUE_MEAN` : the true mean value of human labels.

- `PROXY_MEAN` : the (biased) proxy mean value.

- `N_SEEDS` : the number of simulations we will run in our Monte Carlo experiments.

> **Note on correlation bounds:** Depending on the values of `TRUE_MEAN` and `PROXY_MEAN`, extreme correlation values (close to 0 or 1) may not be achievable. Correlation sweeps are kept within these limits.

Finally, we define the estimation methods that we will compare:

- `True only` : uses uniformly sampled true labels only — the classical CLT gold standard for validity.
- `PPI++` : Prediction-Powered Inference with uniform label sampling and power tuning.
- `ASI` : Active Statistical Inference with oracle label sampling ($\pi_i \propto \text{rms\_error}_i$) and IPW correction.

In [2]:
CONFIDENCE_LEVEL = 0.9  # fixed throughout, 90% Confidence Interval
N_TOTAL = 1000   # total pool size (all records have oracle rms_error)
N_LABELED = 200  # labeling budget (exact for PPI++, expected for ASI with Bernoulli sampling)
TRUE_MEAN = 0.55
PROXY_MEAN = 0.5
N_SEEDS = 1000

METHODS = ['True only', 'PPI++', 'ASI']

correlations = np.arange(0.1, 0.95, 0.1)
correlations_lmh = [0.2, 0.5, 0.8]  # low, medium and high values
corr_labels = ['Low', 'Medium', 'High']

## Data Generation

We use `generate_binary_dataset_with_oracle_sampling` to simulate a realistic evaluation scenario.
It generates `N_TOTAL` records, each containing:
- `y_true` : ground-truth binary label (latent — revealed only for labeled records)
- `y_proxy` : proxy binary prediction (always available)
- `rms_error` : oracle proxy error $\sqrt{\mathbb{E}[(y_{\text{proxy}} - y_{\text{true}})^2 \mid x_i]}$ — quantifies per-record proxy reliability

Records with high `rms_error` are those where the proxy is least reliable. The oracle sampling rule in ASI assigns higher labeling probability to these records:

$$\pi_i = \frac{N_{\text{labeled}} \cdot \text{rms\_error}_i}{\sum_j \text{rms\_error}_j}, \quad \pi_i \in (0, 1]$$

This concentrates the labeling budget on items where true labels add the most information.

The `build_datasets` helper below constructs both an ASI dataset (oracle-sampled labels, with $\pi_i$ attached) and a PPI++ dataset (uniformly-sampled labels) from the same oracle pool, enabling a direct comparison under identical conditions.

In [ ]:
def build_datasets(full_dataset, seed):
    # Build ASI (oracle sampling) and PPI++ (uniform sampling) datasets from the same oracle pool.
    # ASI: each record labeled with prob pi_i proportional to rms_error_i (expected N_LABELED labeled).
    # PPI++: exactly N_LABELED records chosen uniformly at random.
    data = full_dataset.to_numpy(fields=['y_true', 'y_proxy', 'rms_error'])
    y_true_all = data[:, 0].astype(int)
    y_proxy_all = data[:, 1].astype(int)
    rms_error = data[:, 2]
    N = len(full_dataset)

    # Separate RNG for labeling (seeds 100_000+ don't overlap with data gen seeds 0–N_SEEDS)
    rng = np.random.default_rng(seed + 100_000)

    # --- ASI: oracle sampling probabilities ---
    pi = N_LABELED * rms_error / rms_error.sum()
    pi = np.minimum(pi, 1.0)
    labeled_mask_asi = rng.random(N) < pi

    asi_records = []
    for i in range(N):
        rec = {'y_proxy': int(y_proxy_all[i]), 'pi': float(pi[i])}
        if labeled_mask_asi[i]:
            rec['y_true'] = int(y_true_all[i])
        asi_records.append(rec)

    # --- PPI++: uniform random sampling (exactly N_LABELED records) ---
    labeled_idx_ppi = set(rng.choice(N, size=N_LABELED, replace=False))
    ppi_records = []
    for i in range(N):
        if i in labeled_idx_ppi:
            ppi_records.append({'y_true': int(y_true_all[i]), 'y_proxy': int(y_proxy_all[i])})
        else:
            ppi_records.append({'y_proxy': int(y_proxy_all[i])})

    return Dataset(asi_records), Dataset(ppi_records)

In [ ]:
# Single example dataset for illustration (correlation = 0.5)
full_dataset = generate_binary_dataset_with_oracle_sampling(
    N=N_TOTAL,
    true_mean=TRUE_MEAN,
    proxy_mean=PROXY_MEAN,
    correlation=0.5,
    random_seed=42,
)
asi_dataset, ppi_dataset = build_datasets(full_dataset, seed=42)

# Oracle sampling probabilities
pi_example = N_LABELED * full_dataset['rms_error'] / full_dataset['rms_error'].sum()
pi_example = np.minimum(pi_example, 1.0)
pi_uniform = N_LABELED / N_TOTAL

n_labeled_asi = int(np.sum(~np.isnan(asi_dataset['y_true'])))
print(f'Total records : {N_TOTAL}')
print(f'Labeling budget : {N_LABELED}')
print(f'ASI labeled (realized, Bernoulli) : {n_labeled_asi}')
print(f'PPI++ labeled (fixed, uniform) : {N_LABELED}')
print(f'\nOracle pi_i  — min: {pi_example.min():.4f}, max: {pi_example.max():.4f}, mean: {pi_example.mean():.4f}')
print(f'Uniform  pi  — {pi_uniform:.4f} (constant for all records)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pi_example, bins=30, color='darkorange', alpha=0.75, label='ASI (oracle $\pi_i$)')
ax.axvline(pi_uniform, color='steelblue', lw=2.5, linestyle='--', label=f'PPI++ (uniform $\pi$ = {pi_uniform:.3f})')
ax.set_xlabel('Sampling probability $\pi_i$')
ax.set_ylabel('Count')
ax.set_title('Distribution of sampling probabilities: oracle vs uniform  (correlation = 0.5)')
ax.legend()
plt.tight_layout()
plt.show()

The histogram shows that oracle $\pi_i$ values are spread around the uniform value. Records where the proxy is unreliable (high `rms_error`) receive a higher sampling probability, while records where the proxy is already reliable receive a lower one. This is the "active" part of ASI.

## Inference Results

We compare three estimation methods:

| Estimation method | Labels used | Sampling | Notes |
|--------|-----------|----------|-------|
| **True only** | `y_true` (uniform) | Uniform | Classical CLT, gold standard for validity |
| **PPI++** | `y_true` (uniform) + `y_proxy` | Uniform | Valid and efficient; leverages all proxy data |
| **ASI** | `y_true` (oracle) + `y_proxy` | $\pi_i \propto \text{rms\_error}_i$ | Valid, IPW-corrected; focuses budget on uncertain items |

`True only` and `PPI++` use the **same** uniformly selected labeled records. `ASI` uses oracle-selected records with IPW correction.

In [ ]:
def generate_estimates(asi_dataset, ppi_dataset, confidence_level=CONFIDENCE_LEVEL):
    # Return mean and std for all three estimation methods.

    # --- ASI (oracle sampling, IPW-corrected) ---
    asi_result = ASIMeanEstimator().estimate(
        asi_dataset,
        y_true_field='y_true',
        y_proxy_field='y_proxy',
        sampling_probability_field='pi',
        confidence_level=confidence_level,
    )

    # --- PPI++ (uniform sampling) ---
    ppi_result = PPIMeanEstimator().estimate(
        ppi_dataset,
        y_true_field='y_true',
        y_proxy_field='y_proxy',
        confidence_level=confidence_level,
    )

    # --- True only (classical baseline on the uniform labeled subset of ppi_dataset) ---
    # ClassicalMeanEstimator uses nanmean/nanstd and correctly ignores unlabeled records (NaN y_true)
    true_only_result = ClassicalMeanEstimator().estimate(
        ppi_dataset, y_field='y_true', confidence_level=confidence_level
    )

    return {
        'True only': {'mean': true_only_result.mean, 'std': true_only_result.std},
        'PPI++': {
            'mean': ppi_result.mean,
            'std': ppi_result.std,
            'ess': ppi_result.effective_sample_size,
        },
        'ASI': {
            'mean': asi_result.mean,
            'std': asi_result.std,
            'ess': asi_result.effective_sample_size,
        },
    }

In the following sections, we run Monte Carlo experiments to evaluate coverage and confidence interval width.

Each experiment consists of `N_SEEDS` simulations where we generate oracle data, apply each method, and measure the output. This produces `N_SEEDS` samples that we use to estimate empirical coverage and CI width distributions.

The three functions below implement this process:

- `monte_carlo_simulation` : runs `N_SEEDS` simulations for a fixed correlation level, storing mean, std, and ESS per method per seed.
- `compute_hits` : checks whether each seed's confidence interval covers `TRUE_MEAN`.
- `coverage_with_errbar` : estimates the empirical coverage and its own confidence interval via `ClassicalMeanEstimator` on the hit indicators.

In [ ]:
def monte_carlo_simulation(correlation, n_seeds=N_SEEDS):
    # Single Monte Carlo loop: cache per-seed mean, std, and ESS for each estimation method.
    means = {method: np.zeros(n_seeds) for method in METHODS}
    stds = {method: np.zeros(n_seeds) for method in METHODS}
    ess = {method: np.zeros(n_seeds) for method in ['PPI++', 'ASI']}

    for seed in range(n_seeds):
        full_dataset = generate_binary_dataset_with_oracle_sampling(
            N=N_TOTAL,
            true_mean=TRUE_MEAN,
            proxy_mean=PROXY_MEAN,
            correlation=correlation,
            random_seed=seed,
        )
        asi_dataset, ppi_dataset = build_datasets(full_dataset, seed)
        estimates = generate_estimates(asi_dataset, ppi_dataset)

        for method in METHODS:
            means[method][seed] = estimates[method]['mean']
            stds[method][seed] = estimates[method]['std']
        for method in ['PPI++', 'ASI']:
            ess[method][seed] = estimates[method]['ess']

    stats = {method: {'means': means[method], 'stds': stds[method]} for method in METHODS}
    for method in ['PPI++', 'ASI']:
        stats[method]['ess'] = ess[method]
    return stats


def compute_hits(stats, confidence_level):
    # Return per-seed hit indicators {method: float array} at the given confidence level.
    z = norm.ppf((1 + confidence_level) / 2)
    hits = {}
    for method in METHODS:
        lo = stats[method]['means'] - z * stats[method]['stds']
        hi = stats[method]['means'] + z * stats[method]['stds']
        hits[method] = ((lo <= TRUE_MEAN) & (TRUE_MEAN <= hi)).astype(float)
    return hits


def coverage_with_errbar(hits, confidence_level):
    # Estimate empirical coverage + Confidence Interval via ClassicalMeanEstimator
    # on per-seed hit indicators.
    dataset = Dataset([{'hit': h} for h in hits])
    estimator = ClassicalMeanEstimator()
    r = estimator.estimate(dataset, y_field='hit', confidence_level=confidence_level)
    return r.mean, r.confidence_interval.lower_bound, r.confidence_interval.upper_bound

## Coverage Validity

A confidence interval is **valid** if it is built using an estimation method that reliably captures the true value. For example, a 90% confidence interval is valid if, when you repeat the experiment many times, around 90% of those intervals contain the true value.

We check this empirically via Monte Carlo and count how often each method's confidence interval covers `TRUE_MEAN`.

> **Key question:** Does ASI maintain valid coverage under non-uniform oracle sampling, or does the non-uniform selection bias the result?

The IPW correction in ASI is designed to answer: yes, coverage is maintained. The `π_i` denominators de-bias the oracle-selected estimates, recovering the same validity as uniform sampling.

### Coverage vs confidence level for three correlation levels

We sweep the confidence level from 0.55 to 0.95 and plot the observed coverage.
For a valid estimation method, the dots should fall on or above the black diagonal $y = \text{confidence\_level}$.

We do this for **low**, **medium** and **high** proxy correlation.

In [ ]:
# Run Monte Carlo simulations for each correlation level
raw_stats = {correlation: monte_carlo_simulation(correlation) for correlation in correlations}

confidence_levels = np.arange(0.55, 1.00, 0.05)

# Derive coverage for every (correlation, confidence_level) pair
coverages_confidence_intervals = {}
for correlation in correlations_lmh:
    coverages_confidence_intervals[correlation] = {}
    for confidence_level in confidence_levels:
        hits = compute_hits(raw_stats[correlation], confidence_level)
        coverages_confidence_intervals[correlation][confidence_level] = {}
        for method in METHODS:
            coverage_confidence_interval = coverage_with_errbar(hits[method], confidence_level=confidence_level)
            coverages_confidence_intervals[correlation][confidence_level][method] = coverage_confidence_interval

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = {'True only': 'steelblue', 'PPI++': 'forestgreen', 'ASI': 'darkorange'}

for ax, correlation, label in zip(axes, correlations_lmh, corr_labels):
    ax.plot(confidence_levels, confidence_levels, color='black', lw=1.5, linestyle='--', label='Ideal')
    for method in METHODS:
        mean = [coverages_confidence_intervals[correlation][cl][method][0] for cl in confidence_levels]
        lo = [coverages_confidence_intervals[correlation][cl][method][1] for cl in confidence_levels]
        hi = [coverages_confidence_intervals[correlation][cl][method][2] for cl in confidence_levels]

        ax.plot(confidence_levels, mean, marker='o', color=colors[method], label=method)
        ax.fill_between(confidence_levels, lo, hi, alpha=0.15, color=colors[method])

    ax.set_xlabel('Target confidence level')
    ax.set_ylabel('Observed coverage')
    ax.set_title(f'{label} correlation (${correlation}$)')
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.5, 1.0)

fig.suptitle('Empirical coverage vs target confidence level', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**True only**, **PPI++**, and **ASI** all track the diagonal closely, confirming that ASI achieves valid coverage regardless of correlation level. The IPW correction successfully de-biases the oracle-selected estimates.

---

### Coverage vs correlation for fixed confidence level = 0.9

We now fix the confidence level at 90% and vary the proxy-true correlation from 0.1 to 0.9.
This shows that ASI's validity does not degrade as the proxy becomes weaker.

In [ ]:
coverage_by_corr = {}
coverage_ci_by_corr = {}

for correlation in correlations:
    hits = compute_hits(raw_stats[correlation], CONFIDENCE_LEVEL)
    coverage_by_corr[correlation] = {}
    coverage_ci_by_corr[correlation] = {}
    for method in METHODS:
        mean_cov, lo, hi = coverage_with_errbar(hits[method], CONFIDENCE_LEVEL)
        coverage_by_corr[correlation][method] = mean_cov
        coverage_ci_by_corr[correlation][method] = (lo, hi)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for method in METHODS:
    obs = [coverage_by_corr[correlation][method] for correlation in correlations]
    lo = [coverage_ci_by_corr[correlation][method][0] for correlation in correlations]
    hi = [coverage_ci_by_corr[correlation][method][1] for correlation in correlations]
    ax.plot(correlations, obs, marker='o', color=colors[method], label=method)
    ax.fill_between(correlations, lo, hi, alpha=0.15, color=colors[method])

ax.axhline(y=CONFIDENCE_LEVEL, color='red', linestyle='--', lw=2, label=f'Target coverage {CONFIDENCE_LEVEL:.0%}')
ax.set_xlabel('Proxy\u2013true correlation')
ax.set_ylabel('Observed coverage')
ax.set_title(f'Coverage vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})')
ax.set_xlim(0, 1)
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

All three valid methods (**True only**, **PPI++**, **ASI**) remain above the 90% target line across all correlation levels. ASI's IPW correction holds regardless of proxy quality.

---

## Confidence Interval Width

Coverage validity is necessary but not sufficient — we also want **short** intervals. The promise of ASI is that by focusing the labeling budget on uncertain items, oracle sampling extracts more information per label than uniform sampling.

We expect:
- **ASI** to produce the shortest intervals (oracle labeling + proxy correction)
- **PPI++** to be narrower than **True only** (proxy correction helps when correlation is high)
- All three to converge as correlation approaches 0 (proxy carries no signal)

We report the **mean** and the **10th–90th percentile band** to capture variability across Monte Carlo runs.

In [ ]:
z_score = norm.ppf((1 + CONFIDENCE_LEVEL) / 2)
width_by_corr = {
    correlation: {method: 2 * z_score * raw_stats[correlation][method]['stds'] for method in METHODS}
    for correlation in correlations
}

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors_w = {'True only': 'steelblue', 'PPI++': 'forestgreen', 'ASI': 'darkorange'}

for method in METHODS:
    means_w = [np.mean(width_by_corr[correlation][method]) for correlation in correlations]
    q10 = [np.percentile(width_by_corr[correlation][method], 10) for correlation in correlations]
    q90 = [np.percentile(width_by_corr[correlation][method], 90) for correlation in correlations]
    ax.plot(correlations, means_w, marker='o', label=method, color=colors_w[method])
    ax.fill_between(correlations, q10, q90, alpha=0.15, color=colors_w[method])

ax.set_xlabel('Proxy\u2013true correlation')
ax.set_ylabel('Confidence Interval width')
ax.set_title(
    f'Confidence interval width vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})\n'
    'Solid = mean, shaded = 10th\u201390th percentile'
)
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

ASI produces the narrowest intervals, especially at higher correlation levels where the spread in `rms_error` is larger (proxy reliability varies more across records, giving oracle sampling more leverage).

At low correlation, all three methods converge: when the proxy is uniformly bad, there is little to gain from focusing the budget on uncertain items.

---

## Effective Sample Size

The **Effective Sample Size (ESS)** summarises the efficiency gain: it is the number of samples a classical (`True only`) estimator would need to achieve the same confidence interval width.

Since confidence interval width $\propto 1/\sqrt{n}$, ESS is estimated empirically as:

$$\text{ESS} = N_{\text{labeled}} \times \left(\frac{\bar{w}_{\text{True only}}}{\bar{w}_{\text{method}}}\right)^2$$

When ESS $> N_{\text{labeled}}$, the method is more efficient than using labeled data alone.

We compare ASI and PPI++ ESS:
- **PPI++ ESS** $> N_{\text{labeled}}$ whenever the proxy is informative (proxy correction helps)
- **ASI ESS** $>$ **PPI++ ESS** whenever oracle sampling concentrates the budget effectively

In [ ]:
ess_colors = {'PPI++': 'forestgreen', 'ASI': 'darkorange'}

ess_mean = {m: [np.mean(raw_stats[corr][m]['ess']) for corr in correlations] for m in ['PPI++', 'ASI']}
ess_q10 = {m: [np.percentile(raw_stats[corr][m]['ess'], 10) for corr in correlations] for m in ['PPI++', 'ASI']}
ess_q90 = {m: [np.percentile(raw_stats[corr][m]['ess'], 90) for corr in correlations] for m in ['PPI++', 'ASI']}

fig, ax = plt.subplots(figsize=(8, 5))
for method in ['PPI++', 'ASI']:
    ax.plot(correlations, ess_mean[method], marker='o', color=ess_colors[method], label=f'{method} ESS (mean)')
    ax.fill_between(
        correlations, ess_q10[method], ess_q90[method],
        alpha=0.15, color=ess_colors[method], label=f'{method} 10th\u201390th percentile'
    )

ax.axhline(y=N_LABELED, color='steelblue', linestyle='--', lw=2, label=f'Baseline (True only, n={N_LABELED})')
ax.set_xlabel('Proxy\u2013true correlation')
ax.set_ylabel('Effective sample size')
ax.set_title('ASI vs PPI++ effective sample size vs proxy correlation')
ax.set_xlim(0.05, 0.95)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Summary

This notebook has empirically validated that GLIDE's ASI implementation satisfies two key statistical properties:

| Property | Result |
|----------|--------|
| **Coverage validity** | ASI achieves the nominal coverage across all correlation levels and confidence levels tested. The IPW correction successfully de-biases the oracle-selected estimates. |
| **Efficiency over PPI++** | ASI produces shorter confidence intervals than PPI++ when oracle sampling is used. The gain grows with correlation, as higher correlation means more spread in proxy reliability across records — giving oracle sampling more leverage. |

The core mechanism: oracle sampling ($\pi_i \propto \text{rms\_error}_i$) concentrates the labeling budget on the records where the proxy is least reliable. These are the records where a true label adds the most information. Without the IPW correction, this biased selection would invalidate the estimate. With IPW, ASI recovers validity and turns the oracle selection into a pure efficiency gain.

The ESS analysis confirms that at high proxy correlation, ASI can be equivalent to having significantly more labeled data than its nominal budget — a practical gain when true annotation is expensive.